# NeoOLAF DocRED raw-text entity + relation smoke5 run and evaluation

This notebook does the exact smoke setting requested:

1. build a 5-document DocRED slice;
2. run the normal NeoOLAF benchmark runner in parallel;
3. give the model only raw document text plus the global relation vocabulary;
4. do **not** expose source/gold entity IDs to the prompt;
5. evaluate predicted entities and predicted relations against the 5-doc gold slice.

This is different from the previous source-anchored relation adapter. Here NeoOLAF must predict entities from text, then predict relations between those predicted entities.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import re
import shlex
import subprocess
import sys
import time
from pathlib import Path


def find_project_root() -> Path:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidates:
        if (candidate / "experiments/methods/run_neoolaf.py").is_file():
            return candidate
    raise FileNotFoundError("Could not find experiments/methods/run_neoolaf.py. Launch inside NeoOLAF.")

PROJECT_ROOT = find_project_root()
RUNNER_PATH = PROJECT_ROOT / "experiments/methods/run_neoolaf.py"
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
RUNS_DIR = NOTEBOOK_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"RUNNER_PATH={RUNNER_PATH}")
print(f"RUNS_DIR={RUNS_DIR}")


## Configuration

The important flags are:

- `RAW_TEXT_ENTITY_RELATION_MODE = True`
- `SOURCE_ENTITY_ANCHORING = False`
- `DOCRED_DIRECT_CONSTRAINED_EXTRACTION = False`

So this uses full NeoOLAF, not the direct source-anchored relation extractor.

In [ ]:
def first_existing_path(label: str, candidates: list[Path]) -> Path:
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser()
        candidate = candidate if candidate.is_absolute() else (PROJECT_ROOT / candidate)
        candidate = candidate.resolve()
        checked.append(candidate)
        if candidate.is_file():
            print(f"{label}={candidate}")
            return candidate
    raise FileNotFoundError("Could not find " + label + "\n" + "\n".join(str(x) for x in checked))

DATASET_JSONL = first_existing_path(
    "DATASET_JSONL",
    [
        NOTEBOOK_DIR / "../../../ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "RAGTree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT / "ragtree/data/preprocessed/docred_causal.jsonl",
    ],
)

NEOOLAF_ONTOLOGY_PATH = first_existing_path(
    "NEOOLAF_ONTOLOGY_PATH",
    [
        NOTEBOOK_DIR / "../../../ragtree/data/ontology/DocREDOntology/ontology.ttl",
        PROJECT_ROOT.parent / "ragtree/data/ontology/DocREDOntology/ontology.ttl",
        PROJECT_ROOT.parent / "RAGTree/data/ontology/DocREDOntology/ontology.ttl",
        PROJECT_ROOT / "ragtree/data/ontology/DocREDOntology/ontology.ttl",
    ],
)

OPENROUTER_HOST = "https://openrouter.ai/api"
OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY", "")
MODEL_NAME = "openai/gpt-oss-20b"
if not OPENROUTER_KEY:
    raise RuntimeError("Set OPENROUTER_API_KEY in the environment before running this notebook.")

TYPE_FILTER = "dev"
SMOKE_DOCS = 5
DOCUMENT_WORKERS = 2
MAX_WORKERS = 2
MAX_EXPRESSIONS = 80
MAX_RELATION_MENTIONS = 80

RAW_TEXT_ENTITY_RELATION_MODE = True
SOURCE_ENTITY_ANCHORING = False
DOCRED_DIRECT_CONSTRAINED_EXTRACTION = False
FORCE_RELATION_VOCABULARY = True
RELATION_VOCAB_SOURCE = "dataset"

RUN_NAME = "neoolaf_docred_raw_native_er_smoke5"
SMOKE_INPUT_JSONL = RUNS_DIR / f"{RUN_NAME}_input.jsonl"
OUTPUT_JSONL = RUNS_DIR / f"{RUN_NAME}_predictions.canonical.jsonl"
SUMMARY_OUTPUT = RUNS_DIR / f"{RUN_NAME}_run_summary.json"
ERROR_LOG_JSONL = RUNS_DIR / f"{RUN_NAME}_errors.jsonl"
ARTIFACTS_ROOT = RUNS_DIR / f"{RUN_NAME}_artifacts"
RELATION_VOCAB_OUTPUT_PATH = RUNS_DIR / f"{RUN_NAME}_allowed_relations.json"
LOG_PATH = RUNS_DIR / f"{RUN_NAME}.log"
EVAL_JSON = RUNS_DIR / f"{RUN_NAME}_eval.json"
EVAL_CSV = RUNS_DIR / f"{RUN_NAME}_eval_per_doc.csv"

print(f"SMOKE_INPUT_JSONL={SMOKE_INPUT_JSONL}")
print(f"OUTPUT_JSONL={OUTPUT_JSONL}")
print(f"SUMMARY_OUTPUT={SUMMARY_OUTPUT}")
print(f"ERROR_LOG_JSONL={ERROR_LOG_JSONL}")
print(f"ARTIFACTS_ROOT={ARTIFACTS_ROOT}")
print(f"LOG_PATH={LOG_PATH}")


## Build 5-document input slice

The runner receives this 5-document JSONL file, but in raw-text mode the patched runner does not expose source entity IDs to the model prompt.

In [ ]:
def iter_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

selected = []
for rec in iter_jsonl(DATASET_JSONL):
    if TYPE_FILTER == "all" or rec.get("type") == TYPE_FILTER or rec.get("split") == TYPE_FILTER:
        selected.append(rec)
        if len(selected) >= SMOKE_DOCS:
            break

assert len(selected) == SMOKE_DOCS, f"Expected {SMOKE_DOCS} records, got {len(selected)}"
with SMOKE_INPUT_JSONL.open("w", encoding="utf-8") as f:
    for rec in selected:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Wrote {len(selected)} docs to {SMOKE_INPUT_JSONL}")
for rec in selected:
    print(rec.get("document_id") or rec.get("id") or rec.get("title"), "|", rec.get("title"))


## Run NeoOLAF on the 5 raw-text documents

This uses the same `experiments/methods/run_neoolaf.py` runner, in parallel, with native NeoOLAF layers. It does not use the custom raw ER runner and does not use the direct source-anchored extractor.

In [ ]:
for path in [OUTPUT_JSONL, SUMMARY_OUTPUT, ERROR_LOG_JSONL, LOG_PATH, EVAL_JSON, EVAL_CSV]:
    try:
        path.unlink()
    except FileNotFoundError:
        pass

command = [
    sys.executable,
    str(RUNNER_PATH),
    "--dataset-jsonl-path", str(SMOKE_INPUT_JSONL),
    "--ontology-path", str(NEOOLAF_ONTOLOGY_PATH),
    "--output-jsonl-path", str(OUTPUT_JSONL),
    "--summary-output-path", str(SUMMARY_OUTPUT),
    "--error-log-jsonl-path", str(ERROR_LOG_JSONL),
    "--artifacts-root", str(ARTIFACTS_ROOT),
    "--backend-name", "openrouter",
    "--host", OPENROUTER_HOST,
    "--api-key", OPENROUTER_KEY,
    "--model-name", MODEL_NAME,
    "--type-filter", TYPE_FILTER,
    "--relation-vocab-source", RELATION_VOCAB_SOURCE,
    "--relation-vocab-dataset-path", str(DATASET_JSONL),
    "--relation-vocab-ontology-path", str(NEOOLAF_ONTOLOGY_PATH),
    "--relation-vocab-output-path", str(RELATION_VOCAB_OUTPUT_PATH),
    "--force-relation-vocabulary",
    "--raw-text-entity-relation-mode",
    "--output-format", "canonical",
    "--chunk-size", "10000000",
    "--chunk-overlap", "0",
    "--max-chunks", "1",
    "--max-expressions", str(MAX_EXPRESSIONS),
    "--max-relation-mentions", str(MAX_RELATION_MENTIONS),
    "--max-workers", str(MAX_WORKERS),
    "--document-workers", str(DOCUMENT_WORKERS),
    "--temperature", "0.0",
    "--max-tokens", "8192",
    "--openrouter-reasoning-effort", "minimal",
    "--openrouter-exclude-reasoning",
    "--request-timeout", "600",
    "--no-web-search",
    "--disable-wikipedia-lookups",
    "--offline-ontology-only",
    "--no-checkpoints",
    "--no-chunk-checkpoints",
    "--no-resume",
]

if SOURCE_ENTITY_ANCHORING:
    command.append("--source-entity-anchoring")
if DOCRED_DIRECT_CONSTRAINED_EXTRACTION:
    command.append("--docred-direct-constrained-extraction")

print("COMMAND:")
print(" ".join(shlex.quote(x) for x in command))

start = time.time()
with LOG_PATH.open("w", encoding="utf-8") as log:
    proc = subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=log, stderr=subprocess.STDOUT, text=True)
    rc = proc.wait()

elapsed = time.time() - start
print(f"return_code={rc} elapsed_seconds={elapsed:.2f}")
print("Last log lines:")
print("\n".join(LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()[-120:]))

if rc != 0:
    raise RuntimeError(f"NeoOLAF raw native ER smoke5 run failed with return code {rc}")
assert OUTPUT_JSONL.is_file(), f"Prediction file not created: {OUTPUT_JSONL}"
lines = [x for x in OUTPUT_JSONL.read_text(encoding="utf-8").splitlines() if x.strip()]
print(f"prediction_lines={len(lines)}")
assert len(lines) == SMOKE_DOCS, f"Expected {SMOKE_DOCS} prediction lines, got {len(lines)}"


## Inspect predictions

In [ ]:
pred_records = [json.loads(x) for x in OUTPUT_JSONL.read_text(encoding="utf-8").splitlines() if x.strip()]
for rec in pred_records:
    pred = rec.get("prediction", {})
    print("\n" + "="*100)
    print(rec.get("document_id"), "|", rec.get("title"), "parsed_ok=", rec.get("parsed_ok"))
    print("entities:", len(pred.get("entities", [])), "relations:", len(pred.get("relations", [])))
    print("raw_counts:", rec.get("raw_counts", {}))
    print("Entity preview:")
    for ent in pred.get("entities", [])[:20]:
        print("  ", ent.get("label"), "|", ent.get("type"))
    print("Relation preview:")
    for rel in pred.get("relations", [])[:30]:
        print(f"  {rel.get('head')} -- {rel.get('relation')} --> {rel.get('tail')}")


## Evaluate entities and relations

Gold entities are used only here, during evaluation. Predicted entity labels/aliases are mapped to DocRED gold clusters by exact normalized alias matching.

Metrics:

- `entity_inventory`: did the model discover the gold entity clusters?
- `entity_endpoint`: did relation endpoints use gold relation-participating entities?
- `relation_strict`: did `(head_cluster, relation_id, tail_cluster)` match exactly?

In [ ]:
def norm(text) -> str:
    text = str(text or "").strip().lower()
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\s+", " ", text)
    text = text.strip(" \t\n\r\"'`.,;:!?()[]{}")
    return text


def rel_id_from_label(label) -> str:
    label = str(label or "").strip()
    if " : " in label:
        return label.split(" : ", 1)[0].strip()
    return label


def doc_id(record):
    return record.get("document_id") or record.get("id") or record.get("doc_id") or record.get("title")


def gold_entity_table(record):
    entities = record.get("entities") or {}
    table = {}
    if isinstance(entities, dict):
        for eid, ent in entities.items():
            aliases = []
            typ = "entity"
            if isinstance(ent, dict):
                typ = ent.get("type") or typ
                if ent.get("label"):
                    aliases.append(ent.get("label"))
                mentions = ent.get("mentions") or []
                if isinstance(mentions, list):
                    for m in mentions:
                        if isinstance(m, dict):
                            val = m.get("trigger_word") or m.get("name") or m.get("text")
                            if val:
                                aliases.append(val)
            aliases = [str(a).strip() for a in aliases if str(a).strip()]
            table[str(eid)] = {"id": str(eid), "type": str(typ), "aliases": sorted(dict.fromkeys(aliases))}
    elif isinstance(entities, list):
        for i, ent in enumerate(entities):
            if isinstance(ent, dict):
                eid = str(ent.get("id") or ent.get("entity_id") or f"entity_{i:05d}")
                label = ent.get("label") or ent.get("name") or ent.get("text") or eid
                aliases = [label, *(ent.get("aliases") if isinstance(ent.get("aliases"), list) else [])]
                table[eid] = {"id": eid, "type": str(ent.get("type") or "entity"), "aliases": sorted(dict.fromkeys(map(str, aliases)))}
    return table


def gold_alias_index(record):
    table = gold_entity_table(record)
    idx = {}
    for eid, ent in table.items():
        for alias in ent.get("aliases") or []:
            key = norm(alias)
            if key and key not in idx:
                idx[key] = eid
    return idx


def gold_triples(record):
    triples = set()
    relations = record.get("relations") or {}
    if isinstance(relations, dict):
        for rel_label, pairs in relations.items():
            rid = rel_id_from_label(rel_label)
            if not isinstance(pairs, list):
                continue
            for pair in pairs:
                if isinstance(pair, list) and len(pair) >= 2:
                    triples.add((str(pair[0]), rid, str(pair[1])))
                elif isinstance(pair, dict):
                    h = pair.get("head") or pair.get("head_id") or pair.get("subject")
                    t = pair.get("tail") or pair.get("tail_id") or pair.get("object")
                    if h and t:
                        triples.add((str(h), rid, str(t)))
    return triples


def predicted_entity_labels(pred_record):
    labels = []
    for ent in pred_record.get("prediction", {}).get("entities", []) or []:
        if not isinstance(ent, dict):
            continue
        for val in [ent.get("label"), ent.get("name"), ent.get("text"), *(ent.get("aliases") if isinstance(ent.get("aliases"), list) else [])]:
            if val and str(val).strip():
                labels.append(str(val).strip())
    # Relation endpoints are also predicted entities for endpoint/inventory evaluation.
    for rel in pred_record.get("prediction", {}).get("relations", []) or []:
        if isinstance(rel, dict):
            for val in [rel.get("head"), rel.get("tail")]:
                if val and str(val).strip():
                    labels.append(str(val).strip())
    return sorted(dict.fromkeys(labels))


def map_label_to_gold(label, alias_idx):
    key = norm(label)
    if not key:
        return None
    return alias_idx.get(key)


def predicted_entity_set(pred_record, gold_record):
    alias_idx = gold_alias_index(gold_record)
    result = set()
    for label in predicted_entity_labels(pred_record):
        mapped = map_label_to_gold(label, alias_idx)
        result.add(mapped if mapped else f"UNMAPPED::{norm(label)}")
    return result


def predicted_endpoint_set(pred_record, gold_record):
    alias_idx = gold_alias_index(gold_record)
    endpoints = set()
    for rel in pred_record.get("prediction", {}).get("relations", []) or []:
        if not isinstance(rel, dict):
            continue
        for side in ["head", "tail"]:
            label = rel.get(side) or rel.get(f"{side}_id")
            mapped = map_label_to_gold(label, alias_idx)
            endpoints.add(mapped if mapped else f"UNMAPPED::{norm(label)}")
    return endpoints


def predicted_triples(pred_record, gold_record):
    alias_idx = gold_alias_index(gold_record)
    triples = set()
    for rel in pred_record.get("prediction", {}).get("relations", []) or []:
        if not isinstance(rel, dict):
            continue
        h_label = rel.get("head") or rel.get("head_id")
        t_label = rel.get("tail") or rel.get("tail_id")
        r = rel.get("relation_id") or rel_id_from_label(rel.get("relation"))
        h = map_label_to_gold(h_label, alias_idx)
        t = map_label_to_gold(t_label, alias_idx)
        h = h if h else f"UNMAPPED::{norm(h_label)}"
        t = t if t else f"UNMAPPED::{norm(t_label)}"
        if h and r and t:
            triples.add((h, str(r), t))
    return triples


def prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2*p*r/(p+r) if (p+r) else 0.0
    return p, r, f


gold_records = {doc_id(r): r for r in iter_jsonl(SMOKE_INPUT_JSONL)}
pred_records = {doc_id(r): r for r in iter_jsonl(OUTPUT_JSONL)}

metric_sets = {
    "entity_inventory": {"gold": set(), "pred": set()},
    "entity_endpoint": {"gold": set(), "pred": set()},
    "relation_strict": {"gold": set(), "pred": set()},
}
per_doc_rows = []

for did, gold in gold_records.items():
    pred = pred_records.get(did, {"prediction": {"entities": [], "relations": []}})
    gold_entities = set(gold_entity_table(gold).keys())
    pred_entities = predicted_entity_set(pred, gold)
    gold_rels = gold_triples(gold)
    pred_rels = predicted_triples(pred, gold)
    gold_endpoints = {h for h, _, t in gold_rels} | {t for h, _, t in gold_rels}
    pred_endpoints = predicted_endpoint_set(pred, gold)

    local = {
        "document_id": did,
        "title": pred.get("title") or gold.get("title"),
        "parsed_ok": bool(pred.get("parsed_ok", False)),
        "predicted_entity_labels": len(predicted_entity_labels(pred)),
        "predicted_relations": len(pred.get("prediction", {}).get("relations", []) or []),
    }
    for name, g, p in [
        ("entity_inventory", gold_entities, pred_entities),
        ("entity_endpoint", gold_endpoints, pred_endpoints),
        ("relation_strict", gold_rels, pred_rels),
    ]:
        tp = len(g & p)
        fp = len(p - g)
        fn = len(g - p)
        precision, recall, f1 = prf(tp, fp, fn)
        local[f"{name}_gold"] = len(g)
        local[f"{name}_pred"] = len(p)
        local[f"{name}_TP"] = tp
        local[f"{name}_FP"] = fp
        local[f"{name}_FN"] = fn
        local[f"{name}_precision"] = precision
        local[f"{name}_recall"] = recall
        local[f"{name}_f1"] = f1
        metric_sets[name]["gold"] |= {(did, *x) if isinstance(x, tuple) else (did, x) for x in g}
        metric_sets[name]["pred"] |= {(did, *x) if isinstance(x, tuple) else (did, x) for x in p}
    per_doc_rows.append(local)

summary = {}
print("RAW-TEXT NATIVE NEOOLAF ENTITY + RELATION SMOKE5 EVALUATION")
print("="*100)
for name, sets in metric_sets.items():
    g = sets["gold"]
    p = sets["pred"]
    tp = len(g & p)
    fp = len(p - g)
    fn = len(g - p)
    precision, recall, f1 = prf(tp, fp, fn)
    summary[name] = {
        "gold": len(g), "pred": len(p), "TP": tp, "FP": fp, "FN": fn,
        "precision": precision, "recall": recall, "f1": f1,
    }
    print("\n" + name)
    print(f"gold={len(g)} pred={len(p)} TP={tp} FP={fp} FN={fn}")
    print(f"precision={precision:.4f} recall={recall:.4f} f1={f1:.4f}")

EVAL_JSON.write_text(json.dumps({"summary": summary, "per_doc": per_doc_rows}, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
with EVAL_CSV.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(per_doc_rows[0].keys()))
    writer.writeheader()
    writer.writerows(per_doc_rows)

print(f"\nWrote {EVAL_JSON}")
print(f"Wrote {EVAL_CSV}")


## Native KG artifact check

This confirms that full NeoOLAF still produced native graph artifacts for each parsed document.

In [ ]:
def resolve_artifact_dir(raw_path: str) -> Path:
    path = Path(raw_path)
    return path.resolve() if path.is_absolute() else (PROJECT_ROOT / path).resolve()

for rec in pred_records.values():
    print("\n" + "="*100)
    print(rec.get("document_id"), rec.get("title"), "parsed_ok=", rec.get("parsed_ok"))
    artifact_dir = resolve_artifact_dir(rec.get("artifact_dir"))
    print("artifact_dir:", artifact_dir)
    kg_local = list(artifact_dir.rglob("kg_local.json"))
    kg_inferred = list(artifact_dir.rglob("kg_inferred.json"))
    print("kg_local found:", len(kg_local))
    print("kg_inferred found:", len(kg_inferred))
    if kg_local:
        print("  ", kg_local[0])
    if kg_inferred:
        print("  ", kg_inferred[0])
